**(PT_BR)**
# 🕵️‍♂️ SISTEMA DE INSPEÇÃO VISUAL AUTOMATIZADO COM IA PARA DETECÇÃO DE FALHAS EM VIDROS LAMINADOS

Projeto desenvolvido para a disciplina de **Projeto Integrado 1/2**, para a faculdade **IFSP - Instituto Federal de São Paulo**

**Discentes:**  
- Almir Rodrigues da Silva
- Gabriel Rodrigues de Novaes
- Pedro Henrique Canôas da Silva

**Orientador:**  
- Ivan Lucas Arantes

**Coorientador:**  
- Valdeci Donizete Gonçalves

**Resumo**
A proposta deste trabalho consiste no desenvolvimento, análise e aplicação de um sistema de inspeção visual automatizado utilizando Inteligência Artificial (IA), baseado na metodologia YOLO11, para a detecção de falhas em vidros laminados. O objetivo é identificar defeitos de forma rápida, precisa e em tempo real, otimizando o controle de qualidade no processo de fabricação. Ao longo deste trabalho, serão abordados aspectos relacionados ao desenvolvimento completo da solução, incluindo a programação do sistema, o treinamento do modelo de IA, a validação com imagens reais de vidros com e sem falhas, bem como técnicas de aprimoramento do modelo. Espera-se uma análise comparativa dos resultados obtidos em termos de assertividade e velocidade de processamento, considerando variações na quantidade de imagens utilizadas e nos ciclos de treinamento, com destaque para as características de desempenho alcançadas e sua viabilidade para aplicação industrial.

**(EN_US)**
# 🕵️‍♂️ AUTOMATED VISUAL INSPECTION SYSTEM USING ARTIFICIAL INTELLIGENCE FOR DETECTING DEFECTS IN LAMINATED GLASS

Project developed for the **Integrated Project 1/2** course at the **Federal Institute of São Paulo (IFSP)**.

**Students:**  
- Almir Rodrigues da Silva
- Gabriel Rodrigues de Novaes
- Pedro Henrique Canôas da Silva

**Advisor:**  
- Ivan Lucas Arantes

**Co-advisor:**  
- Valdeci Donizete Gonçalves

**Resumo**
This study presents the development and implementation of an automated visual inspection system based on Artificial Intelligence (AI) for defect detection in laminated glass. The system utilizes the YOLO11 object detection framework to identify surface defects in real time with high accuracy. The proposed approach covers the entire development pipeline, including system programming, model training, and validation using real images of defective and non-defective glass. Model optimization techniques are also applied to enhance detection performance. A comparative analysis is conducted to evaluate accuracy and processing speed across different image quantities and training cycles. The results demonstrate the system's potential for industrial application, highlighting its effectiveness in improving quality control in the glass manufacturing process.

# 🚀 Training, Validation, and Prediction - YOLO11

In [ ]:
# === Imports ===
!pip install ultralytics -q  # Install the Ultralytics YOLO package silently

import os
import time
from datetime import datetime
from ultralytics import YOLO
import ultralytics

# === Configuration ===

# Path to the pretrained model used as the starting point (YOLOv11m in this case)
MODEL_PATH = 'yolo26s.pt'

# Path to the dataset configuration file (defines train/val/test splits and class names)
DATA_YAML = '/kaggle/input/glass-defect/config.yaml'

# Base dataset directory (same level as this notebook)
dataset_dir = '/kaggle/input/glass-defect/dataset/dataset'

# Training hyperparameters
EPOCHS = 1000           # Maximum number of training epochs
PATIENCE = 300         # Stop training if no improvement after this many epochs
CONF_TRAIN = 0.4       # Minimum confidence threshold during training
CONF_PREDICT = 0.7     # Minimum confidence threshold for predictions
SINGLE_CLASS = True    # Set to True if training a single-class detection model

# Directory paths
IMAGE_TO_PREDICT_DIR = os.path.join(dataset_dir, 'images', 'test')  # Test image folder
RESULT_DIR = os.path.join('/kaggle/working/runs', 'results')        # Output folder for results

In [ ]:
# === Environment Check ===
print("🔍 Checking YOLO environment...")
ultralytics.checks()  # Verify that all YOLO dependencies and configurations are properly installed

print(f"📦 loading moodel: {MODEL_PATH}")
model = YOLO(MODEL_PATH)

In [ ]:
# --- Training ---
print("🚀 Starting model training...")

train_results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    patience=PATIENCE,
    val=True,
    batch=-1,              # ou um valor maior que 12, ex: 24
    single_cls=SINGLE_CLASS,
    lr0=0.002,
    device='cuda',
    amp=True,
    cache='ram',            # ou 'disk' se o dataset for grande
    workers=8,               # ajuste conforme sua CPU
)

In [ ]:
# --- Validation ---
print("📈 Evaluating model on validation set...")

val_results = model.val(
    conf=CONF_TRAIN,   # Minimum confidence threshold to consider a detection valid
    save_json=True     # Save metrics in COCO JSON format
)

In [ ]:
# --- Prediction ---
print(f"🖼️ Searching for images in: {IMAGE_TO_PREDICT_DIR}")

# Filter valid image extensions
image_extensions = ('.jpg', '.jpeg', '.png', '.bmp')
image_paths = [
    os.path.join(IMAGE_TO_PREDICT_DIR, f)
    for f in os.listdir(IMAGE_TO_PREDICT_DIR)
    if f.lower().endswith(image_extensions)
]

if not image_paths:
    print(f"❌ No images found in {IMAGE_TO_PREDICT_DIR}")
else:
    print(f"🔍 {len(image_paths)} images found for prediction.")
    pred_results = model.predict(source=image_paths, conf=CONF_PREDICT)

In [ ]:
# --- Save Results ---
os.makedirs(RESULT_DIR, exist_ok=True)

for i, result in enumerate(pred_results):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"result_{i+1}_{timestamp}.jpg"
    result_path = os.path.join(RESULT_DIR, filename)
    result.save(filename=result_path)
    print(f"💾 Result saved at: {result_path}")

print("✅ Execution completed successfully!")

# 🎯 YOLO11 Real-Time Defect Detection from Camera Stream
Detect defects in real-time using a trained YOLO11 model from an IP camera stream


In [ ]:
# --- Imports and Basic Constants ---
import cv2
import time
import threading
import numpy as np
from ultralytics import YOLO
import os
from IPython.display import clear_output
import matplotlib.pyplot as plt

# Path to trained YOLO model
# MODEL_PATH = 'best.pt'
# IP camera URL
CAMERA_URL = 'http://put.your.ip:here/video' 

# Initialize YOLO model
model = YOLO(MODEL_PATH)

# Global variables for threading
running = True
annotated_frame = None
frame_for_inference = None
lock = threading.Lock()

In [ ]:
# --- Inference Thread Function ---
def inference_thread():
    """
    Continuously runs inference on the latest frame captured from the camera.
    Annotates defects and updates the global annotated_frame variable.
    """
    global frame_for_inference, annotated_frame, running

    while running:
        with lock:
            # Get the latest frame for inference
            frame = frame_for_inference.copy() if frame_for_inference is not None else None
            frame_for_inference = None

        if frame is not None:
            # Run prediction
            results = model.predict(source=frame, conf=0.5, verbose=False)[0]

            # Annotate defects on the frame
            annotated = frame.copy()
            for box in results.boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                cv2.rectangle(annotated, (x1, y1), (x2, y2), (0, 0, 255), 2)
                cv2.putText(
                    annotated, "DEFECT", (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2
                )

            # Update the global annotated frame
            with lock:
                annotated_frame = annotated

        # Small delay to reduce CPU/GPU usage
        time.sleep(0.01)

In [ ]:
# --- Camera Initialization and Thread Start ---
cap = cv2.VideoCapture(CAMERA_URL)
if not cap.isOpened():
    raise RuntimeError("❌ Failed to access the IP camera.")

# Start the inference thread
thread = threading.Thread(target=inference_thread)
thread.start()

print("✅ Camera connected. Press Ctrl+C to stop.")

In [ ]:
# --- Main Loop: Display Stream with Real-Time Annotations ---
try:
    while True:
        # Capture frame from camera
        ret, frame = cap.read()
        if not ret:
            print("⚠️ Failed to capture frame.")
            break

        # Provide frame to inference thread
        with lock:
            frame_for_inference = frame.copy()
            display_frame = annotated_frame if annotated_frame is not None else frame

        # Convert BGR to RGB for proper display in matplotlib
        display_frame_rgb = cv2.cvtColor(display_frame, cv2.COLOR_BGR2RGB)

        # Display in notebook
        clear_output(wait=True)
        plt.imshow(display_frame_rgb)
        plt.axis('off')
        plt.show()

        # Delay to avoid saturating GPU/CPU
        time.sleep(0.1)

except KeyboardInterrupt:
    print("🛑 Manually interrupted.")

finally:
    # --- Cleanup ---
    running = False
    thread.join()
    cap.release()
    plt.close('all')
    print("👋 Finished.")
